# Kaggle Predict F1 Pit Stops: Preprocessing and EDA

This notebook is the first pass over the competition data. It is designed to run on Kaggle with data mounted at `/kaggle/input/competitions/playground-series-s6e5`, while also supporting a local `data/` directory.

Goals:
- Load `train.csv`, `test.csv`, and `sample_submission.csv`.
- Check schema, missing values, cardinality, duplicates, and target balance.
- Explore numerical and categorical feature relationships with `PitNextLap`.
- Compare train/test distributions to catch drift before modeling.
- Build reusable preprocessing helpers that avoid target leakage.

## 1. Setup

In [ ]:
import os
import gc
import warnings
from pathlib import Path

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 120)

sns.set_theme(style="whitegrid", palette="tab10")
RANDOM_STATE = 42

In [ ]:
def find_data_dir() -> Path:
    candidates = [
        Path("/kaggle/input/competitions/playground-series-s6e5"),
        Path("/kaggle/input/playground-series-s6e5"),
        Path("../input/competitions/playground-series-s6e5"),
        Path("../input/playground-series-s6e5"),
        Path("data"),
        Path("../data"),
        Path("."),
    ]
    for path in candidates:
        if (path / "train.csv").exists() and (path / "test.csv").exists():
            return path
    raise FileNotFoundError("Could not find train.csv and test.csv. Update DATA_DIR manually.")


DATA_DIR = find_data_dir()
OUTPUT_DIR = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DATA_DIR, OUTPUT_DIR

## 2. Load Data

In [ ]:
def reduce_memory_usage(df: pd.DataFrame) -> pd.DataFrame:
    """Downcast numeric columns to make repeated EDA/modeling faster on Kaggle."""
    out = df.copy()
    for col in out.columns:
        col_type = out[col].dtype
        if pd.api.types.is_integer_dtype(col_type):
            out[col] = pd.to_numeric(out[col], downcast="integer")
        elif pd.api.types.is_float_dtype(col_type):
            out[col] = pd.to_numeric(out[col], downcast="float")
        elif pd.api.types.is_object_dtype(col_type):
            nunique = out[col].nunique(dropna=False)
            if nunique / max(len(out), 1) < 0.5:
                out[col] = out[col].astype("category")
    return out


train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

train = reduce_memory_usage(train)
test = reduce_memory_usage(test)

print(f"train shape: {train.shape}")
print(f"test shape: {test.shape}")
print(f"sample submission shape: {sample_submission.shape}")
train.head()

In [ ]:
TARGET = "PitNextLap"
ID_COL = "id"

feature_cols = [c for c in train.columns if c not in [TARGET]]
categorical_cols = train[feature_cols].select_dtypes(include=["object", "category"]).columns.tolist()
numeric_cols = [c for c in feature_cols if c not in categorical_cols]

print("Target:", TARGET)
print("Categorical columns:", categorical_cols)
print("Numeric columns:", numeric_cols)

## 3. Schema and Data Quality Checks

In [ ]:
def dataframe_overview(df: pd.DataFrame, name: str) -> pd.DataFrame:
    overview = pd.DataFrame({
        "dataset": name,
        "dtype": df.dtypes.astype(str),
        "missing": df.isna().sum(),
        "missing_pct": df.isna().mean().mul(100),
        "unique": df.nunique(dropna=False),
        "memory_mb": df.memory_usage(deep=True).div(1024 ** 2),
    })
    return overview.sort_values(["missing_pct", "unique"], ascending=[False, False])


overview = pd.concat([
    dataframe_overview(train, "train"),
    dataframe_overview(test, "test"),
])
overview

In [ ]:
quality_checks = pd.DataFrame({
    "check": [
        "train duplicated rows",
        "test duplicated rows",
        "train duplicated id",
        "test duplicated id",
        "train/test id overlap",
    ],
    "value": [
        int(train.duplicated().sum()),
        int(test.duplicated().sum()),
        int(train[ID_COL].duplicated().sum()) if ID_COL in train else np.nan,
        int(test[ID_COL].duplicated().sum()) if ID_COL in test else np.nan,
        int(pd.Index(train[ID_COL]).intersection(test[ID_COL]).shape[0]) if ID_COL in train and ID_COL in test else np.nan,
    ],
})
quality_checks

In [ ]:
train.describe(include="all").T

## 4. Target Balance

`PitNextLap` is the target. We expect a class imbalance because most laps are not followed by a pit stop.

In [ ]:
target_counts = train[TARGET].value_counts().sort_index()
target_rate = train[TARGET].mean()

display(pd.DataFrame({"count": target_counts, "pct": target_counts / len(train)}))
print(f"Positive rate: {target_rate:.4f}")

fig, ax = plt.subplots(figsize=(5, 4))
sns.countplot(data=train, x=TARGET, ax=ax)
ax.set_title("Target distribution")
ax.bar_label(ax.containers[0])
plt.show()

## 5. Categorical Features

In [ ]:
def categorical_summary(df: pd.DataFrame, columns: list[str], target: str) -> pd.DataFrame:
    rows = []
    for col in columns:
        stats = df.groupby(col, observed=True)[target].agg(["count", "mean"]).reset_index()
        stats["feature"] = col
        stats = stats.rename(columns={col: "value", "mean": "target_rate"})
        stats["global_rate_diff"] = stats["target_rate"] - df[target].mean()
        rows.append(stats[["feature", "value", "count", "target_rate", "global_rate_diff"]])
    return pd.concat(rows, ignore_index=True) if rows else pd.DataFrame()


cat_summary = categorical_summary(train, categorical_cols, TARGET)
cat_summary.sort_values(["feature", "count"], ascending=[True, False]).head(40)

In [ ]:
for col in categorical_cols:
    top_values = train[col].value_counts().head(15).index
    plot_df = train[train[col].isin(top_values)].copy()
    order = plot_df.groupby(col, observed=True)[TARGET].mean().sort_values(ascending=False).index
    
    fig, axes = plt.subplots(1, 2, figsize=(14, 4))
    sns.countplot(data=plot_df, y=col, order=top_values, ax=axes[0])
    axes[0].set_title(f"Top {col} values by count")
    
    sns.barplot(data=plot_df, y=col, x=TARGET, order=order, estimator=np.mean, errorbar=None, ax=axes[1])
    axes[1].axvline(train[TARGET].mean(), color="red", linestyle="--", linewidth=1, label="global rate")
    axes[1].set_title(f"{col}: target rate")
    axes[1].legend()
    plt.tight_layout()
    plt.show()

## 6. Numerical Features

The raw description suggests a few long-tailed lap-time and degradation features. We inspect both distributions and target-rate trends.

In [ ]:
numeric_for_eda = [c for c in numeric_cols if c != ID_COL]

num_summary = train[numeric_for_eda + [TARGET]].agg(["count", "mean", "std", "min", "median", "max"]).T
num_summary["skew"] = train[numeric_for_eda + [TARGET]].skew(numeric_only=True)
num_summary

In [ ]:
n_cols = 3
n_rows = int(np.ceil(len(numeric_for_eda) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, max(4, 3 * n_rows)))
axes = np.ravel(axes)

for ax, col in zip(axes, numeric_for_eda):
    sns.histplot(train[col], bins=60, kde=False, ax=ax)
    ax.set_title(col)

for ax in axes[len(numeric_for_eda):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
def binned_target_rate(df: pd.DataFrame, col: str, target: str, bins: int = 20) -> pd.DataFrame:
    tmp = df[[col, target]].copy()
    tmp["bin"] = pd.qcut(tmp[col], q=bins, duplicates="drop")
    out = tmp.groupby("bin", observed=True)[target].agg(["count", "mean"]).reset_index()
    out["bin_mid"] = out["bin"].apply(lambda x: x.mid).astype(float)
    return out


n_cols = 3
n_rows = int(np.ceil(len(numeric_for_eda) / n_cols))
fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, max(4, 3 * n_rows)))
axes = np.ravel(axes)

for ax, col in zip(axes, numeric_for_eda):
    rate_df = binned_target_rate(train, col, TARGET, bins=20)
    sns.lineplot(data=rate_df, x="bin_mid", y="mean", marker="o", ax=ax)
    ax.axhline(train[TARGET].mean(), color="red", linestyle="--", linewidth=1)
    ax.set_title(f"{col}: binned target rate")
    ax.set_ylabel("PitNextLap rate")

for ax in axes[len(numeric_for_eda):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

## 7. Outlier Review

A few fields can contain extreme race-event values. For tree models, clipping is often optional. For linear models, neural nets, or scaled distance-based models, clipping can help.

In [ ]:
def outlier_table(df: pd.DataFrame, columns: list[str], q_low: float = 0.001, q_high: float = 0.999) -> pd.DataFrame:
    rows = []
    for col in columns:
        low = df[col].quantile(q_low)
        high = df[col].quantile(q_high)
        rows.append({
            "feature": col,
            "min": df[col].min(),
            f"q{q_low}": low,
            "median": df[col].median(),
            f"q{q_high}": high,
            "max": df[col].max(),
            "below_low": int((df[col] < low).sum()),
            "above_high": int((df[col] > high).sum()),
        })
    return pd.DataFrame(rows)


outlier_table(train, numeric_for_eda).sort_values("above_high", ascending=False)

## 8. Train/Test Distribution Drift

Synthetic competition datasets can have subtle shifts. This section compares category coverage and numeric distributions.

In [ ]:
def category_coverage(train_df: pd.DataFrame, test_df: pd.DataFrame, columns: list[str]) -> pd.DataFrame:
    rows = []
    for col in columns:
        train_values = set(train_df[col].dropna().astype(str))
        test_values = set(test_df[col].dropna().astype(str))
        rows.append({
            "feature": col,
            "train_unique": len(train_values),
            "test_unique": len(test_values),
            "test_only": len(test_values - train_values),
            "train_only": len(train_values - test_values),
        })
    return pd.DataFrame(rows)


category_coverage(train, test, categorical_cols)

In [ ]:
def psi(train_s: pd.Series, test_s: pd.Series, bins: int = 20, eps: float = 1e-6) -> float:
    quantiles = np.linspace(0, 1, bins + 1)
    edges = np.unique(train_s.quantile(quantiles).to_numpy())
    if len(edges) <= 2:
        edges = np.linspace(min(train_s.min(), test_s.min()), max(train_s.max(), test_s.max()), bins + 1)
    edges[0] = -np.inf
    edges[-1] = np.inf
    train_pct = pd.cut(train_s, edges, include_lowest=True).value_counts(normalize=True, sort=False).to_numpy()
    test_pct = pd.cut(test_s, edges, include_lowest=True).value_counts(normalize=True, sort=False).to_numpy()
    train_pct = np.clip(train_pct, eps, None)
    test_pct = np.clip(test_pct, eps, None)
    return float(np.sum((test_pct - train_pct) * np.log(test_pct / train_pct)))


drift_rows = []
for col in numeric_for_eda:
    drift_rows.append({
        "feature": col,
        "train_mean": train[col].mean(),
        "test_mean": test[col].mean(),
        "mean_diff": test[col].mean() - train[col].mean(),
        "train_std": train[col].std(),
        "test_std": test[col].std(),
        "psi": psi(train[col], test[col]),
    })

drift = pd.DataFrame(drift_rows).sort_values("psi", ascending=False)
drift

In [ ]:
top_drift_cols = drift.head(6)["feature"].tolist()

for col in top_drift_cols:
    fig, ax = plt.subplots(figsize=(8, 4))
    sns.kdeplot(train[col], label="train", fill=True, alpha=0.25, ax=ax)
    sns.kdeplot(test[col], label="test", fill=True, alpha=0.25, ax=ax)
    ax.set_title(f"Train/test distribution: {col}")
    ax.legend()
    plt.show()

## 9. Feature Engineering Starter Set

This starter set uses only row-level information available in both train and test. It avoids target encoding here because target encoders need careful cross-validation to avoid leakage.

The competition notes say `Normalized_TyreLife` was removed. We should be careful with features that reconstruct it too directly. The engineered ratios below are included for exploration, but later model validation should decide whether they generalize.

In [ ]:
def add_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    eps = 1e-6
    
    if {"LapNumber", "RaceProgress"}.issubset(out.columns):
        out["EstimatedRaceLaps"] = out["LapNumber"] / out["RaceProgress"].clip(lower=eps)
        out["EstimatedLapsRemaining"] = out["EstimatedRaceLaps"] - out["LapNumber"]
        out["LapNumber_x_RaceProgress"] = out["LapNumber"] * out["RaceProgress"]
    
    if {"TyreLife", "LapNumber"}.issubset(out.columns):
        out["TyreLife_to_LapNumber"] = out["TyreLife"] / out["LapNumber"].clip(lower=eps)
    
    if {"TyreLife", "EstimatedRaceLaps"}.issubset(out.columns):
        out["TyreLife_to_EstimatedRaceLaps"] = out["TyreLife"] / out["EstimatedRaceLaps"].clip(lower=eps)
    
    if {"LapTime (s)", "LapTime_Delta"}.issubset(out.columns):
        out["LapTime_plus_Delta"] = out["LapTime (s)"] + out["LapTime_Delta"]
        out["AbsLapTime_Delta"] = out["LapTime_Delta"].abs()
    
    if {"Position", "Position_Change"}.issubset(out.columns):
        out["PreviousPositionApprox"] = out["Position"] - out["Position_Change"]
        out["AbsPosition_Change"] = out["Position_Change"].abs()
    
    if "Compound" in out.columns:
        out["IsSoft"] = (out["Compound"].astype(str) == "SOFT").astype("int8")
        out["IsMedium"] = (out["Compound"].astype(str) == "MEDIUM").astype("int8")
        out["IsHard"] = (out["Compound"].astype(str) == "HARD").astype("int8")
        out["IsWetOrIntermediate"] = out["Compound"].astype(str).isin(["WET", "INTERMEDIATE"]).astype("int8")
    
    return reduce_memory_usage(out)


train_fe = add_features(train)
test_fe = add_features(test)

new_cols = [c for c in train_fe.columns if c not in train.columns]
print("New features:", new_cols)
train_fe[new_cols].head()

In [ ]:
engineered_numeric = [c for c in new_cols if pd.api.types.is_numeric_dtype(train_fe[c])]

if engineered_numeric:
    n_cols = 3
    n_rows = int(np.ceil(len(engineered_numeric) / n_cols))
    fig, axes = plt.subplots(n_rows, n_cols, figsize=(16, max(4, 3 * n_rows)))
    axes = np.ravel(axes)
    
    for ax, col in zip(axes, engineered_numeric):
        rate_df = binned_target_rate(train_fe, col, TARGET, bins=20)
        sns.lineplot(data=rate_df, x="bin_mid", y="mean", marker="o", ax=ax)
        ax.axhline(train_fe[TARGET].mean(), color="red", linestyle="--", linewidth=1)
        ax.set_title(f"{col}: target rate")
    
    for ax in axes[len(engineered_numeric):]:
        ax.axis("off")
    
    plt.tight_layout()
    plt.show()

## 10. Preprocessing Pipeline

This pipeline is intentionally simple and model-agnostic. Tree boosting libraries can often consume categoricals differently, but this scikit-learn pipeline is useful for baselines and for creating clean validation splits.

In [ ]:
def make_onehot_encoder() -> OneHotEncoder:
    try:
        return OneHotEncoder(handle_unknown="ignore", min_frequency=20, sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", min_frequency=20, sparse=True)


def make_preprocessor(df: pd.DataFrame, target: str = TARGET, id_col: str = ID_COL) -> tuple[ColumnTransformer, list[str], list[str]]:
    drop_cols = [target]
    # Keep id out of most models. It is sequential and usually not causal.
    if id_col in df.columns:
        drop_cols.append(id_col)
    
    X = df.drop(columns=[c for c in drop_cols if c in df.columns])
    cat_cols = X.select_dtypes(include=["object", "category"]).columns.tolist()
    num_cols = [c for c in X.columns if c not in cat_cols]
    
    numeric_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler()),
    ])
    categorical_pipe = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", make_onehot_encoder()),
    ])
    
    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_pipe, num_cols),
            ("cat", categorical_pipe, cat_cols),
        ],
        remainder="drop",
        sparse_threshold=0.3,
    )
    return preprocessor, num_cols, cat_cols


preprocessor, model_numeric_cols, model_categorical_cols = make_preprocessor(train_fe)
print(f"Model numeric columns ({len(model_numeric_cols)}): {model_numeric_cols}")
print(f"Model categorical columns ({len(model_categorical_cols)}): {model_categorical_cols}")

In [ ]:
X = train_fe.drop(columns=[TARGET])
y = train_fe[TARGET].astype("int8")

X_train, X_valid, y_train, y_valid = train_test_split(
    X,
    y,
    test_size=0.2,
    stratify=y,
    random_state=RANDOM_STATE,
)

print(X_train.shape, X_valid.shape)
print("train positive rate:", y_train.mean())
print("valid positive rate:", y_valid.mean())

In [ ]:
# Fit preprocessing only on training fold to avoid validation leakage.
X_train_processed = preprocessor.fit_transform(X_train)
X_valid_processed = preprocessor.transform(X_valid)

print("Processed train shape:", X_train_processed.shape)
print("Processed valid shape:", X_valid_processed.shape)

## 11. Save Prepared Artifacts

These files are optional convenience outputs. For full modeling, it is usually better to recompute features inside each cross-validation fold.

In [ ]:
try:
    train_fe.to_parquet(OUTPUT_DIR / "train_preprocessed.parquet", index=False)
    test_fe.to_parquet(OUTPUT_DIR / "test_preprocessed.parquet", index=False)
    saved_paths = [OUTPUT_DIR / "train_preprocessed.parquet", OUTPUT_DIR / "test_preprocessed.parquet"]
except Exception as exc:
    print(f"Parquet save failed ({exc}). Falling back to CSV.")
    train_fe.to_csv(OUTPUT_DIR / "train_preprocessed.csv", index=False)
    test_fe.to_csv(OUTPUT_DIR / "test_preprocessed.csv", index=False)
    saved_paths = [OUTPUT_DIR / "train_preprocessed.csv", OUTPUT_DIR / "test_preprocessed.csv"]

print("Saved:")
for path in saved_paths:
    print(path)

## 12. Notes for the Modeling Notebook

- Evaluation is likely probability-based, so optimize validation AUC/log loss depending on the competition metric.
- Use stratified folds because `PitNextLap` is imbalanced.
- Treat high-cardinality `Driver` carefully. One-hot is fine for linear baselines, while CatBoost or target encoding may help later.
- Check whether engineered tyre-life ratio features generalize. They may be powerful, but they can also overfit the synthetic generation process.
- Compare models with and without the original F1 strategy dataset only after this baseline EDA is stable.